# A scratchpad for designing custom vega/altair themes
Preparing modules and loading data from `vega-datasets`:

In [71]:
import polars as pl
import altair as alt
import importlib
import custom_altair_themes
from vega_datasets import data

# importlib.reload(custom_altair_themes)

alt.theme.enable('custom')
alt.renderers.enable("html")

'''
The following configuration allows for plotting >5000 rows of
data from a DataFrame (regardless of DataFrame used). However,
this will not be necessary for the example data used (install
vegafusion if needed)
'''
# alt.data_transformers.enable("vegafusion") # required for large datasets (rows > 5000)

df_sw = pl.DataFrame(data.seattle_weather())
df_cars = pl.DataFrame(data.cars())

## Plotting the data
Starting by examining the data and its schema.

In [3]:
print(df_sw.schema)
print(df_cars.schema)

Schema([('date', Datetime(time_unit='ns', time_zone=None)), ('precipitation', Float64), ('temp_max', Float64), ('temp_min', Float64), ('wind', Float64), ('weather', String)])
Schema([('Name', String), ('Miles_per_Gallon', Float64), ('Cylinders', Int64), ('Displacement', Float64), ('Horsepower', Float64), ('Weight_in_lbs', Int64), ('Acceleration', Float64), ('Year', Datetime(time_unit='ns', time_zone=None)), ('Origin', String)])


### Choosing the data
It is sensible to plot `temp_min` and `temp_max` against one another for `seattle_weather`, as those features will demonstrate strong linear correlation.

For `cars`, it would be most sensible to plot `Miles_per_Gallon` and `Horsepower` against one another, as those features will demonstrate an inverse correlation.

In [ ]:
plot_sw_temps = alt.Chart(df_sw).mark_point(
  size = 40,
  color = "black",
  strokeWidth = 1,
  opacity = 0.9
  ).encode(
    x = alt.X("temp_max", title = "Maximum daily temperature (°C)"),
    y = alt.Y("temp_min", title = "Minimum daily temperature (°C)"),
    fill = alt.Color("temp_max").scale().title("Max temp. (°C)"),
    tooltip = ["date", "temp_max", "temp_min"],
    ).interactive()
  
plot_cars_mpg_hpwr = alt.Chart(df_cars).mark_point(
  size = 40,
  color = "black",
  strokeWidth = 1,
  opacity = 0.9
  ).encode(
    x = alt.X("Horsepower", title = "Horsepower (# of horses)"),
    y = alt.Y("Miles_per_Gallon", title = "Miles per gallon (mpg)"),
    fill = alt.Color("Horsepower:Q").scale().title("Horsepower"), # encoded fill breaks regression line for some reason
    tooltip = ["Name", "Horsepower", "Miles_per_Gallon"],
    ).interactive()

plot = (plot_sw_temps | plot_cars_mpg_hpwr).resolve_scale(fill = "independent") # required for separate color scales when concatening
plot

alt.HConcatChart(...)

### Theming the data
Now the plots should be themed.

In [34]:
# custom_altair_themes.options(grid = False, darkmode = False, topAndRightBorder=False)

# let's try with stripped plots
stripped_plot_sw_temps = alt.Chart(df_sw).mark_point(strokeWidth = 0.25).encode(
  x = alt.X("temp_max", title = "Maximum daily temperature (°C)"),
  y = alt.Y("temp_min", title = "Minimum daily temperature (°C)"),
  fill = alt.Color("temp_max").title("Max temp. (°C)"),
  tooltip = ["date", "temp_max", "temp_min"],
  ).properties(title = {"text": "Main title", "subtitle": "Subtitle"}).interactive()
stripped_plot_cars_hpwr_mpg = alt.Chart(df_cars).mark_point(size = 10).encode(
  x = alt.X("Horsepower", title = "Horsepower (# of horses)"),
  y = alt.Y("Miles_per_Gallon", title = "Miles per gallon (mpg)"),
  fill = alt.Color("Horsepower").scale(scheme = "purplegreen").title("Horsepower"),
  tooltip = ["Name", "Horsepower", "Miles_per_Gallon"]
  ).properties(title = {"text": "Main title", "subtitle": "Subtitle"}).interactive()

# stripped_plot_sw_temps
plot = ((plot_sw_temps | plot_cars_mpg_hpwr).resolve_scale(fill = "independent")
        & (stripped_plot_sw_temps | stripped_plot_cars_hpwr_mpg).resolve_scale(fill = "independent"))
# alt.Chart.save(plot, './output/plot.html', inline=True, embed_options={'renderer':'svg'})
alt.Chart.save(plot, "./output/plot.png", ppi = 600)
plot

NameError: name 'plot_sw_temps' is not defined

### Boxplots
Preparing a theme for boxplots.

In [132]:
import custom_altair_themes
custom_altair_themes.options(angledX = True, topAndRightBorder=False, grid = False, darkmode = True, markFillOpacity = 1, markStrokeColor = "black")
importlib.reload(custom_altair_themes) # required for reload
alt.theme.enable('custom') # required for reload

# boxplot = alt.Chart(df_sw).mark_boxplot().encode(
boxplot = alt.Chart(df_sw).mark_boxplot().encode(
  x = alt.X("weather:N", title = "Weather", scale = alt.Scale(domain = ["sun", "drizzle", "fog", "rain", "snow"])),
  y = alt.Y("temp_max:Q", title = "Max temp. (°C)"),
  color = alt.Color("weather:O", title = "Weather", scale = alt.Scale(domain = ["sun", "drizzle", "fog", "rain", "snow"])),
)

points = alt.Chart(df_sw).mark_circle().encode(
  # x = alt.X("weather"),
  x = alt.X("weather:N", title = "Weather", scale = alt.Scale(domain = ["sun", "drizzle", "fog", "rain", "snow"])),
  y = alt.Y("temp_max:Q", axis = alt.Axis(translate = 0)),
  xOffset = alt.XOffset('offset:Q'),
  color = alt.Color("weather:O", title = "Weather", scale = alt.Scale(domain = ["sun", "drizzle", "fog", "rain", "snow"])),
).transform_calculate(offset = 'sqrt(-2*log(random()))*cos(2*PI*random())')

# y0 = alt.Chart(df_sw).mark_rule().encode(y = alt.datum(0))

final = (points + boxplot).properties(width = 125, height = 125).configure_axis(titleFontSize = 11, labelFontSize = 11)
alt.Chart.save(final, "./output/plot.png", ppi = 1200)
final.interactive()

alt.LayerChart(...)

### Ramping heatmap
Preparing a theme for ramping heatmaps.

In [71]:
custom_altair_themes.options(markFillOpacity = 1, ticks = False)
importlib.reload(custom_altair_themes) # required for reload
alt.theme.enable('custom') # required for reload

heatmap = (
    alt.Chart(df_sw)
    .mark_rect()
    .encode(
        x=alt.X("month(date):O", title="Month"),
        y=alt.Y("weather:N", title=None),
        color=alt.Color("count():Q", title="Frequency", scale = alt.Scale(scheme = "categoricalSequential")),
        tooltip=["month(date):O", "weather:N", "count():Q"],
    )
    .properties(width=300, height=200, title="Seattle Weather Heatmap")
).interactive()

alt.Chart.save(heatmap, "./output/plot.png", ppi = 1200)
heatmap

SchemaValidationError: 'categoricalSequential' is an invalid value for `scheme`. Valid values are:

- One of ['accent', 'category10', 'category20', 'category20b', 'category20c', 'dark2', 'paired', 'pastel1', 'pastel2', 'set1', 'set2', 'set3', 'tableau10', 'tableau20']
- One of ['blues', 'tealblues', 'teals', 'greens', 'browns', 'greys', 'purples', 'warmgreys', 'reds', 'oranges']
- One of ['turbo', 'viridis', 'inferno', 'magma', 'plasma', 'cividis', 'bluegreen', 'bluegreen-3', 'bluegreen-4', 'bluegreen-5', 'bluegreen-6', 'bluegreen-7', 'bluegreen-8', 'bluegreen-9', 'bluepurple', 'bluepurple-3', 'bluepurple-4', 'bluepurple-5', 'bluepurple-6', 'bluepurple-7', 'bluepurple-8', 'bluepurple-9', 'goldgreen', 'goldgreen-3', 'goldgreen-4', 'goldgreen-5', 'goldgreen-6', 'goldgreen-7', 'goldgreen-8', 'goldgreen-9', 'goldorange', 'goldorange-3', 'goldorange-4', 'goldorange-5', 'goldorange-6', 'goldorange-7', 'goldorange-8', 'goldorange-9', 'goldred', 'goldred-3', 'goldred-4', 'goldred-5', 'goldred-

### Diverging heatmap
Preparing a theme for diverging heatmaps.

In [ ]:
custom_altair_themes.options(ticks = False, verticalY = True, darkmode = True, markFillOpacity = 1.0)
importlib.reload(custom_altair_themes) # required for reload
alt.theme.enable('custom') # required for reload

heatmap = (
    alt.Chart(df_sw)
    .mark_rect()
    .encode(
        x=alt.X("month(date):O", title="Month"),
        y=alt.Y("year(date):O", title="Year"),
        color=alt.Color(
            "mean(temp_max):Q",
            scale=alt.Scale(domainMid=18),  # Midpoint for divergence
            title="Avg Max Temp (°F)"
        ),
        tooltip=["month(date):Q", "year(date):O", "mean(temp_max):Q"],
    )
    .properties(width=500, height=300, title="Diverging Heatmap of Max Temperature in Seattle")
).interactive()

alt.Chart.save(heatmap, "./output/plot.png", ppi = 1200)
heatmap

alt.Chart(...)

### Bar charts
Preparing a theme for bar charts.

In [75]:
custom_altair_themes.options(angledX = True, topAndRightBorder = True, darkmode = False, markFillOpacity = 1.0)
importlib.reload(custom_altair_themes) # required for reload
alt.theme.enable('custom') # required for reload

bars = alt.Chart(df_sw).mark_bar().encode(
    x=alt.X('yearmonth(date):O', title='Month'),
    y=alt.Y('average(precipitation):Q', title='Avg Precipitation (inches)'),
    color=alt.Color('weather:N', title='Weather Type')
).properties(
    title='Seattle Weather: Average Monthly Precipitation'
)

alt.Chart.save(bars, "./output/plot.png", ppi = 1200)
bars

alt.Chart(...)

### Area marks
Preparing a theme for area marks in `altair`.

In [4]:
custom_altair_themes.options(angledX = True, topAndRightBorder = True, darkmode = False, fontWeight = 300)
importlib.reload(custom_altair_themes) # required for reload
alt.theme.enable('custom') # required for reload

df_stocks = data.stocks.url

area = alt.Chart(df_stocks).transform_filter(alt.datum.symbol == "GOOG").mark_area(
    # line={"color": "darkgreen"},
    # line = True,
    # point = True,
    color=alt.Gradient(
        gradient="linear",
        stops=[
            alt.GradientStop(color="white", offset=0),
            alt.GradientStop(color="black", offset=1),
        ],
        x1=1,
        x2=1,
        y1=1,
        y2=0,
    ),
).encode(
    alt.X("date:T"),
    alt.Y("price:Q"),
)

alt.Chart.save(area, "./output/plot.png", ppi = 1200)
area

alt.Chart(...)